# ThreatLens AI — Notebook 08: Time-Series Threat Forecasting

**Stage in the pipeline:** `AI Engines -> Forecasting`

### What this notebook does
1. Loads the cleaned dataset and aggregates it into hourly time windows
2. Builds lag / rolling-window / calendar features -- the blueprint's recommended feature family for this task
3. Trains a gradient-boosted regressor to predict next-hour attack ratio
4. Evaluates with MAE / RMSE (the blueprint's stated metrics)
5. Recursively forecasts the 1h / 6h / 24h horizons shown on the dashboard's Threat Forecast page
6. Plots the forecast curve, matching the dashboard's probability chart

Needs a `timestamp` column -- same honesty note as Notebooks 06/07: if missing, the cell below explains exactly what to do.


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.forecasting import (
    require_timestamp_column, build_time_series, make_supervised_features,
    train_forecaster, evaluate_forecaster, forecast_horizon,
)

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (11, 5)
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


In [ ]:
df = pd.read_parquet(PROCESSED_DIR / "cicids2017_cleaned.parquet")
ts_col = require_timestamp_column(df)
print(f"Using '{ts_col}' as the time column")

## 1. Aggregate into an hourly time series

Row-level events get collapsed into one row per hour: total event count, attack event count, and the attack ratio -- the value we're actually forecasting. Forecasting individual rows wouldn't make sense (network flows aren't a smooth signal); forecasting the *rate* of attacks per time window is what the blueprint's 1h/6h/24h horizons are actually about.


In [ ]:
series = build_time_series(df, ts_col, attack_col="attack_category", freq="1h")
print(f"Built {len(series):,} hourly buckets")
series.head(10)

In [ ]:
plt.figure()
plt.plot(series.index, series["attack_ratio"], color="#00ffa3")
plt.title("Hourly attack ratio over time")
plt.ylabel("Attack ratio")
plt.tight_layout()
plt.show()

## 2. Build supervised learning features

`make_supervised_features()` creates lag features (attack ratio N hours ago), a rolling mean/std (recent trend), and calendar features (hour of day, day of week) -- turning the raw time series into a table a standard regressor can learn from.


In [ ]:
feat = make_supervised_features(series, target_col="attack_ratio", n_lags=6)
print(f"Feature table: {feat.shape[0]} rows x {feat.shape[1]} columns")
feat.head()

## 3. Train/test split — chronological, not random

For time series, we split by TIME (earlier data trains, later data tests) rather than a random shuffle. A random split would let the model "see the future" via nearby rows in the training set, giving a misleadingly good score that wouldn't hold up in real forecasting.


In [ ]:
feature_cols = [c for c in feat.columns if c not in ("total_events", "attack_events", "attack_ratio")]
split_idx = int(len(feat) * 0.8)

train, test = feat.iloc[:split_idx], feat.iloc[split_idx:]
X_train, y_train = train[feature_cols], train["attack_ratio"]
X_test, y_test = test[feature_cols], test["attack_ratio"]

print(f"Train: {len(train)} hours | Test: {len(test)} hours")

## 4. Train and evaluate

MAE and RMSE, per the blueprint's stated evaluation metrics for this engine.


In [ ]:
model = train_forecaster(X_train, y_train)
y_pred = model.predict(X_test)

metrics = evaluate_forecaster(y_test, y_pred)
print(f"MAE:  {metrics['mae']:.4f}")
print(f"RMSE: {metrics['rmse']:.4f}")

In [ ]:
plt.figure()
plt.plot(y_test.index, y_test.values, label="Actual", color="#3aa0ff")
plt.plot(y_test.index, y_pred, label="Predicted", color="#ff4d5a", linestyle="--")
plt.legend()
plt.title("Forecast vs actual — held-out test period")
plt.ylabel("Attack ratio")
plt.tight_layout()
plt.show()

## 5. Forecast the 1h / 6h / 24h horizons

Starting from the single most recent row of real features, `forecast_horizon()` predicts forward recursively -- each predicted value becomes the newest lag for the next prediction, exactly matching the dashboard's "Next 1 Hour / 6 Hours / 24 Hours" cards.


In [ ]:
last_row = feat.iloc[-1]
predictions_24h = forecast_horizon(model, last_row, steps=24, n_lags=6)

horizon_1h = predictions_24h[0]
horizon_6h = np.mean(predictions_24h[:6])
horizon_24h = np.mean(predictions_24h)

print(f"Next 1 hour  -- predicted attack ratio: {horizon_1h:.1%}")
print(f"Next 6 hours -- avg predicted attack ratio: {horizon_6h:.1%}")
print(f"Next 24 hours -- avg predicted attack ratio: {horizon_24h:.1%}")

In [ ]:
plt.figure()
hours_ahead = list(range(1, 25))
plt.plot(hours_ahead, predictions_24h, marker="o", color="#00ffa3", markersize=4)
plt.axvline(1, color="#7f9c93", linestyle=":", alpha=0.6)
plt.axvline(6, color="#7f9c93", linestyle=":", alpha=0.6)
plt.axvline(24, color="#7f9c93", linestyle=":", alpha=0.6)
plt.xlabel("Hours ahead")
plt.ylabel("Predicted attack ratio")
plt.title("24-hour threat probability forecast")
plt.tight_layout()
plt.show()

## 6. Summary & next steps

| Item | Result |
|---|---|
| Time bucket size | 1 hour |
| MAE / RMSE | see Section 4 |
| Next 1h / 6h / 24h forecast | see Section 5 |

**Honest limitation:** this forecast is only as good as the historical pattern it learned from — a genuinely novel attack campaign that doesn't resemble anything in the training window won't be predicted well by this (or any) time-series model. That's exactly why forecasting is meant to sit *alongside* the anomaly detector and classifier (Notebooks 02-03), not replace them — the three engines catch different kinds of blind spots in each other.

This completes every AI engine in the blueprint: **Anomaly Detection, Attack Classification, Threat Scoring, UBA, Attack Graph, SHAP, and Forecasting.** What's left is wiring them together — the FastAPI + PostgreSQL + Redis backend, delivered as the next set of files alongside this notebook.
